## Introduction

A universal differential equation (UDE) combines known mechanisms with a
trainable function. Instead of asking a neural network to reproduce the complete
dynamics, we give it a narrower job: correct a specific part of a mechanistic
model that we know is incomplete.

This is the classical residual, or "missing physics," UDE construction: a known
mechanistic vector field is retained while a universal approximator learns only
an unknown or misspecified contribution
([Rackauckas et al., 2020](https://doi.org/10.48550/arXiv.2001.04385)). In
neuroscience, this hybrid formulation provides a bridge between interpretable
neural dynamics and flexible data-driven models
([El-Gazzar and van Gerven, 2025](https://doi.org/10.3389/fncom.2025.1677930)).
Unlike a fully learned Neural ODE, where a neural network parameterizes the
complete vector field ([Chen et al., 2018](https://proceedings.neurips.cc/paper/2018/hash/69386f6bb1dfed68692a24c8686939b9-Abstract.html)),
the MLP here augments equations whose known structure remains explicit.

This tutorial constructs a reproducible stochastic four-region FitzHugh--Nagumo
network ([FitzHugh, 1961](https://doi.org/10.1016/S0006-3495(61)86902-6);
[Nagumo et al., 1962](https://doi.org/10.1109/JRPROC.1962.288235)). The synthetic
teacher uses the complete cubic voltage damping, while the incomplete model
contains only half of it. One Equinox multilayer perceptron (MLP) supplies the
missing local correction at every region:

$$
\begin{aligned}
\mathrm dV_i &= \left[V_i - \alpha\frac{V_i^3}{3} - W_i + I + C_i
            + g_\theta(V_i, W_i)\right]\mathrm dt + \sigma\,\mathrm dB_i, \\
\mathrm dW_i &= \frac{V_i + a - bW_i}{\tau}\,\mathrm dt,
\end{aligned}
$$

where $\alpha=1/2$ in the incomplete model, $C_i$ is the known network input,
and $B_i$ is an independent Brownian motion for each region. The target
correction remains

$$
g^*(V_i) = -(1-\alpha)\frac{V_i^3}{3}.
$$

The same $g_\theta$ is evaluated independently at all four nodes. Their different
initial conditions and network inputs expose it to several trajectories through
state space, but there is still only one set of trainable MLP weights.

::: {.callout-note}
## Why retain half of the cubic term?

Removing all cubic damping makes the incomplete FitzHugh--Nagumo system
unbounded before an initially untrained MLP can correct it. Retaining a known
stabilizing component is both numerically safer and a more realistic UDE setup:
the mechanistic model is approximate rather than structurally unstable.
:::

In [ ]:
try:
    import google.colab
    print("Running in Google Colab - installing dependencies...")
    !pip install -q tvboptim
    print("✓ Dependencies installed!")
except ImportError:
    pass

In [ ]:

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import time

jax.config.update("jax_enable_x64", True)

from tvboptim.experimental.network_dynamics import (
    Bunch,
    DenseGraph,
    Network,
    prepare,
)
from tvboptim.experimental.network_dynamics.coupling import LinearCoupling
from tvboptim.experimental.network_dynamics.dynamics import AbstractDynamics
from tvboptim.experimental.network_dynamics.noise import AdditiveNoise
from tvboptim.experimental.network_dynamics.solvers import Heun
from tvboptim.optim import OptaxOptimizer
from tvboptim.types import (
    EquinoxParameter,
    combine_state,
    partition_state,
)

## Complete and Incomplete Dynamics

The teacher contains the complete cubic term. It is used only to generate a
small synthetic data set whose ground truth is known.

In [ ]:

class TeacherFitzHughNagumo(AbstractDynamics):
    STATE_NAMES = ("V", "W")
    INITIAL_STATE = (-1.0, -0.5)
    DEFAULT_PARAMS = Bunch(a=0.7, b=0.8, tau=12.5, I=0.5)
    COUPLING_INPUTS = {"structural": 1}

    def dynamics(self, t, state, params, coupling, external):
        del t, external
        V, W = state
        network_input = coupling.structural[0]
        dV = V - V**3 / 3.0 - W + params.I + network_input
        dW = (V + params.a - params.b * W) / params.tau
        return jnp.stack((dV, dW))

The UDE retains half of the cubic damping and calls the same correction module
once per node. TVB-Optim presents the state to `dynamics` in state-major form,
`[states, nodes]`, so both `V` and `W` below have shape `[nodes]`. The Equinox
MLP, however, describes the correction at one node: with `in_size=2`, it expects
one feature vector `[V_i, W_i]`. Stacking along the last axis therefore builds a
`[nodes, 2]` array whose rows are the inputs for the individual nodes. For this
two-state example, `state.T` would have the same values; constructing
`node_features` explicitly documents which states the learned term sees and in
which order, and generalizes naturally to selected or derived features.

`jax.vmap` then applies the *same* MLP to every row of `node_features`. This is
shared-weight evaluation, not four copies of the MLP. Because the MLP was
created with `out_size=1`, one evaluation returns shape `[1]` rather than a JAX
scalar. The mapped result consequently has shape `[nodes, 1]`, and `[:, 0]`
removes that singleton output dimension to obtain the `[nodes]` correction
required by `base_drift`. Without this indexing, adding `[nodes, 1]` to
`[nodes]` would trigger broadcasting and produce an unintended two-dimensional
array.

In [ ]:

class UDEFitzHughNagumo(AbstractDynamics):
    STATE_NAMES = ("V", "W")
    INITIAL_STATE = (-1.0, -0.5)
    DEFAULT_PARAMS = Bunch(
        a=0.7,
        b=0.8,
        tau=12.5,
        I=0.5,
        cubic_fraction=0.5,
        correction=None,
    )
    COUPLING_INPUTS = {"structural": 1}

    def dynamics(self, t, state, params, coupling, external):
        del t, external
        V, W = state
        network_input = coupling.structural[0]

        node_features = jnp.stack((V, W), axis=-1)  # [nodes, 2]
        learned_term = jax.vmap(params.correction)(node_features)[:, 0]

        base_drift = (
            V
            - params.cubic_fraction * V**3 / 3.0
            - W
            + params.I
            + network_input
        )
        dW = (V + params.a - params.b * W) / params.tau
        return jnp.stack((base_drift + learned_term, dW))

## Four Coupled Regions, One Shared Correction

We use a small directed graph. TVB-Optim stores dense connectivity as
`weights[target, source]`, so each row lists the incoming connection strengths
for one target region. The graph is asymmetric to give the four otherwise
identical regions different network contexts.

In [ ]:

weights = jnp.array(
    [
        [0.0, 0.8, 0.0, 0.2],
        [0.1, 0.0, 0.7, 0.0],
        [0.4, 0.0, 0.0, 0.6],
        [0.0, 0.3, 0.2, 0.0],
    ]
)

initial_state = jnp.array(
    [
        [-1.2, -0.8, 0.1, 0.7],  # V
        [-0.6, -0.2, 0.3, 0.5],  # W
    ]
)

DT = 0.05
T1 = 15.0
COUPLING_GAIN = 0.12
NOISE_SIGMA = 0.08
TRAIN_NOISE_KEY = jax.random.key(11)
N_NODES = weights.shape[0]
REGION_LABELS = tuple(f"R{node + 1}" for node in range(N_NODES))

In [ ]:

angles = np.linspace(0, 2 * np.pi, N_NODES, endpoint=False) + np.pi / 4
positions = np.stack((np.cos(angles), np.sin(angles)), axis=1)

fig, (ax_graph, ax_shared) = plt.subplots(1, 2, figsize=(8.1, 3.4))

for target in range(N_NODES):
    for source in range(N_NODES):
        strength = float(weights[target, source])
        if strength == 0.0:
            continue
        start = positions[source] * 0.82
        end = positions[target] * 0.82
        ax_graph.annotate(
            "",
            xy=end,
            xytext=start,
            arrowprops=dict(
                arrowstyle="->",
                color="0.35",
                alpha=0.35 + 0.65 * strength,
                linewidth=0.8 + 2.0 * strength,
                shrinkA=12,
                shrinkB=12,
            ),
        )

colors = plt.cm.cividis(np.linspace(0.15, 0.85, N_NODES))
for node, (position, color) in enumerate(zip(positions, colors)):
    ax_graph.scatter(*position, s=650, color=color, edgecolor="black", zorder=3)
    ax_graph.text(
        *position,
        REGION_LABELS[node],
        ha="center",
        va="center",
        fontsize=9,
    )

ax_graph.set_title("Directed mechanistic coupling")
ax_graph.set_aspect("equal")
ax_graph.set_xlim(-1.35, 1.35)
ax_graph.set_ylim(-1.25, 1.25)
ax_graph.axis("off")

ax_shared.axis("off")
for node, color in enumerate(colors):
    y = 0.83 - node * 0.2
    ax_shared.text(
        0.06,
        y,
        f"$(V_{node + 1}, W_{node + 1})$",
        transform=ax_shared.transAxes,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.35", facecolor=color, edgecolor="black"),
    )
    ax_shared.annotate(
        "",
        xy=(0.61, 0.5),
        xytext=(0.22, y),
        xycoords="axes fraction",
        arrowprops=dict(arrowstyle="->", color="0.35"),
    )

ax_shared.text(
    0.72,
    0.5,
    "$g_\\theta(V,W)$\nshared weights",
    transform=ax_shared.transAxes,
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.8", facecolor="white", edgecolor="black", linewidth=1.5),
)
ax_shared.set_title("One module, evaluated at every region")

plt.tight_layout()

## Generate Reproducible Noisy Training Data

Both networks use the same graph, coupling, initial conditions, solver, and
mechanistic parameters. A small additive process noise acts only on voltage.
Their only intentional structural difference is the missing half of the cubic
voltage damping.

In [ ]:

def prepare_network(dynamics, noise_key=TRAIN_NOISE_KEY):
    network = Network(
        dynamics=dynamics,
        coupling={
            "structural": LinearCoupling(source="V", G=COUPLING_GAIN)
        },
        graph=DenseGraph(weights),
        noise=AdditiveNoise(
            sigma=NOISE_SIGMA,
            apply_to="V",
            key=noise_key,
        ),
    )
    solve_fn, config = prepare(
        network,
        Heun(),
        t0=0.0,
        t1=T1,
        dt=DT,
    )
    config.initial_state.dynamics = initial_state
    return solve_fn, config


teacher_solve, teacher_config = prepare_network(TeacherFitzHughNagumo())
target_solution = teacher_solve(teacher_config)

assert target_solution.variable_names == ("V", "W")
assert target_solution.ys.shape == (int(T1 / DT), 2, N_NODES)
assert jnp.all(jnp.isfinite(target_solution.ys))

The teacher and UDE use the same fixed random key. This *common random numbers*
design makes the example reproducible and prevents the optimizer from comparing
different random forcing on every evaluation. The noise realization is part of
the synthetic training data; only the MLP weights change during fitting.

This is a software demonstration with synthetic observations, not evidence that
the learned term is a biologically identified mechanism. Its advantage is that
we know exactly what the correction should recover.

## Add an Equinox Parameter

We initialize a small MLP and set its final layer to zero. The first incomplete
rollout therefore contains no learned correction and exposes the mechanistic
model's misspecification directly.

In [ ]:

def make_zero_initialized_mlp(key, width_size=16, depth=2):
    module = eqx.nn.MLP(
        in_size=2,
        out_size=1,
        width_size=width_size,
        depth=depth,
        activation=jax.nn.tanh,
        key=key,
    )
    final_layer = module.layers[-1]
    return eqx.tree_at(
        lambda model: (
            model.layers[-1].weight,
            model.layers[-1].bias,
        ),
        module,
        (
            jnp.zeros_like(final_layer.weight),
            jnp.zeros_like(final_layer.bias),
        ),
    )


correction = EquinoxParameter(
    make_zero_initialized_mlp(jax.random.key(7))
)
ude_solve, ude_config = prepare_network(
    UDEFitzHughNagumo(correction=correction)
)

print(f"Correction wrapper: {type(ude_config.dynamics.correction).__name__}")
print(f"Shared module: {type(ude_config.dynamics.correction.module).__name__}")

`EquinoxParameter` marks the MLP's inexact array leaves as trainable. Its
activation functions, architecture, and other non-array leaves remain static.
The wrapper itself is callable, so the dynamics does not need to unwrap it.

Because an Equinox module contains callable leaves, a full configuration should
cross the compilation boundary through `eqx.filter_jit`:

In [ ]:

compiled_solve = eqx.filter_jit(ude_solve)

initial_solution = ude_solve(ude_config)
compiled_initial = compiled_solve(ude_config)

assert jnp.allclose(compiled_initial.ys, initial_solution.ys)
assert jnp.all(jnp.isfinite(initial_solution.ys))

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(8.1, 5.2), sharex=True, sharey=True)
for node, (ax, color) in enumerate(zip(axes.flat, colors)):
    ax.plot(
        target_solution.ts,
        target_solution.ys[:, 0, node],
        "k--",
        linewidth=1.8,
        label="complete teacher",
    )
    ax.plot(
        initial_solution.ts,
        initial_solution.ys[:, 0, node],
        color=color,
        linewidth=1.4,
        label="incomplete model",
    )
    ax.set_title(REGION_LABELS[node])
    ax.grid(alpha=0.2)

axes[0, 0].legend(frameon=False, fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Time [a.u.]")
for ax in axes[:, 0]:
    ax.set_ylabel("V")
plt.tight_layout()

## Verify Gradient Flow

Partitioning separates the MLP arrays from the fixed graph, solver state,
mechanistic parameters, and callable activation leaves. A direct gradient through
the rollout confirms that every trainable leaf receives finite signal.

In [ ]:

target_V = target_solution.ys[:, 0, :]

def loss(config):
    predicted_V = ude_solve(config).ys[:, 0, :]
    return jnp.mean((predicted_V - target_V) ** 2)


diff_config, static_config = partition_state(ude_config)

def partitioned_loss(diff):
    return loss(combine_state(diff, static_config))


gradient = jax.grad(partitioned_loss)(diff_config)
gradient_leaves = jax.tree.leaves(gradient)
gradient_norm = jnp.sqrt(sum(jnp.sum(leaf**2) for leaf in gradient_leaves))

assert gradient_leaves
assert all(jnp.all(jnp.isfinite(leaf)) for leaf in gradient_leaves)
assert gradient_norm > 0.0

print(f"Trainable array leaves: {len(gradient_leaves)}")
print(f"Initial gradient norm: {float(gradient_norm):.3e}")

## Fit the Shared Neural Correction

`OptaxOptimizer` uses the same API for ordinary `Parameter` values and
`EquinoxParameter` modules. We record one loss value per ten-step optimizer chunk
to keep the notebook output and Python overhead small.

In [ ]:

compiled_loss = eqx.filter_jit(loss)
initial_loss = float(compiled_loss(ude_config))
loss_steps = [0]
loss_values = [initial_loss]

def record_loss(step, diff, static, fitting_data, aux, loss_value, grads):
    del fitting_data, aux, grads
    loss_steps.append(int(step) + 1)
    loss_values.append(float(loss_value))
    return False, diff, static


optimizer = OptaxOptimizer(
    loss,
    optax.adam(learning_rate=3e-3),
    callback=record_loss,
)
fitted_config, _ = optimizer.run(
    ude_config,
    max_steps=500,
    chunk_size=10,
)

final_loss = float(compiled_loss(fitted_config))
loss_values[-1] = final_loss
fitted_solution = compiled_solve(fitted_config)

assert jnp.all(jnp.isfinite(fitted_solution.ys))
assert final_loss < initial_loss * 0.05

In [ ]:

fig, (ax_loss, ax_fit) = plt.subplots(1, 2, figsize=(8.1, 3.4))

ax_loss.semilogy(loss_steps, loss_values, color="black", marker="o", markersize=3)
ax_loss.set_xlabel("Optimizer step")
ax_loss.set_ylabel("Voltage trajectory MSE")
ax_loss.set_title(f"Loss: {initial_loss:.3f} → {final_loss:.4f}")
ax_loss.grid(alpha=0.25, which="both")

for node, color in enumerate(colors):
    ax_fit.plot(
        target_solution.ts,
        target_solution.ys[:, 0, node],
        "--",
        color=color,
        alpha=0.55,
        linewidth=2.2,
    )
    ax_fit.plot(
        fitted_solution.ts,
        fitted_solution.ys[:, 0, node],
        color=color,
        linewidth=1.0,
        label=REGION_LABELS[node],
    )

ax_fit.set_xlabel("Time [a.u.]")
ax_fit.set_ylabel("V")
ax_fit.set_title("Dashed target, solid fitted")
ax_fit.grid(alpha=0.2)
ax_fit.legend(frameon=False, fontsize=7, ncol=2)

plt.tight_layout()

## What Did the MLP Learn?

The target residual depends only on voltage, although the MLP receives both
`V` and `W`. We evaluate it at every state visited by the teacher and compare it
with the known missing half-cubic term.

In [ ]:

V_observed = target_solution.ys[:, 0, :].reshape(-1)
W_observed = target_solution.ys[:, 1, :].reshape(-1)
observed_features = jnp.stack((V_observed, W_observed), axis=-1)

learned_residual = jax.vmap(fitted_config.dynamics.correction)(
    observed_features
)[:, 0]
true_residual = -(1.0 - fitted_config.dynamics.cubic_fraction) * V_observed**3 / 3.0
residual_correlation = np.corrcoef(
    np.asarray(learned_residual), np.asarray(true_residual)
)[0, 1]

In [ ]:

fig, ax = plt.subplots(figsize=(6.2, 3.8))

n_times = target_solution.ys.shape[0]
for node, color in enumerate(colors):
    indices = np.arange(node, n_times * N_NODES, N_NODES)
    ax.scatter(
        np.asarray(V_observed)[indices],
        np.asarray(learned_residual)[indices],
        s=9,
        alpha=0.3,
        color=color,
        label=REGION_LABELS[node],
    )

V_grid = jnp.linspace(V_observed.min(), V_observed.max(), 300)
true_curve = -(1.0 - fitted_config.dynamics.cubic_fraction) * V_grid**3 / 3.0
ax.plot(V_grid, true_curve, color="black", linewidth=2.2, label="true residual")
ax.set_xlabel("V")
ax.set_ylabel("Correction to $\\dot V$")
ax.set_title(f"Residual correlation: r = {residual_correlation:.3f}")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()

## Validation Initial Conditions

The fitted module has seen four related trajectories, not the entire phase
plane. We therefore rerun both models from new node states without changing the
MLP or graph, and replace the training key with a new shared noise key. This
checks interpolation to nearby trajectories and another random realization; it
is not a claim of global extrapolation.

In [ ]:

validation_initial_state = jnp.array(
    [
        [-1.05, -0.45, 0.35, 0.95],
        [-0.45, 0.05, 0.40, 0.70],
    ]
)

validation_teacher_config = teacher_config.copy()
validation_teacher_config.initial_state.dynamics = validation_initial_state
validation_fitted_config = fitted_config.copy()
validation_fitted_config.initial_state.dynamics = validation_initial_state

validation_noise_key = jax.random.key(29)
validation_teacher_config.noise.key = validation_noise_key
validation_fitted_config.noise.key = validation_noise_key

validation_target = teacher_solve(validation_teacher_config)
validation_prediction = compiled_solve(validation_fitted_config)
validation_mse = jnp.mean(
    (validation_prediction.ys[:, 0, :] - validation_target.ys[:, 0, :]) ** 2
)

assert jnp.isfinite(validation_mse)
assert validation_mse < initial_loss * 0.1

In [ ]:

fig, ax = plt.subplots(figsize=(8.1, 3.4))
for node, color in enumerate(colors):
    ax.plot(
        validation_target.ts,
        validation_target.ys[:, 0, node],
        "--",
        color=color,
        alpha=0.6,
        linewidth=2.1,
    )
    ax.plot(
        validation_prediction.ts,
        validation_prediction.ys[:, 0, node],
        color=color,
        linewidth=1.0,
        label=REGION_LABELS[node],
    )

ax.set_xlabel("Time [a.u.]")
ax.set_ylabel("V")
ax.set_title(f"Validation voltage MSE = {float(validation_mse):.4f}")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8, ncol=4)
plt.tight_layout()

## Explore Network Capacity and Cost

The width of 16 used above is a deliberately modest default, not a previously
optimized hyperparameter. We can make that choice more systematic by holding
the depth, optimizer, step count, training trajectory, and validation trajectory
fixed while varying only the hidden width.

Here we use a simple tolerance rule: choose the smallest network whose
validation MSE is within 20% of the best value in the sweep. Unlike selecting
the absolute minimum, this favors a smaller model when additional capacity
provides only a marginal improvement.

In [ ]:

WIDTHS = (2, 4, 8, 16, 32)
SWEEP_DEPTH = 2
SWEEP_STEPS = 500
KNEE_TOLERANCE = 0.20

def parameter_count(module):
    arrays = jax.tree.leaves(eqx.filter(module, eqx.is_inexact_array))
    return sum(array.size for array in arrays)


def block_until_ready(tree):
    for leaf in jax.tree.leaves(tree):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()


capacity_results = []
for width in WIDTHS:
    module_key = jax.random.fold_in(jax.random.key(101), width)
    candidate_correction = EquinoxParameter(
        make_zero_initialized_mlp(
            module_key,
            width_size=width,
            depth=SWEEP_DEPTH,
        )
    )
    candidate_solve, candidate_config = prepare_network(
        UDEFitzHughNagumo(correction=candidate_correction)
    )

    def candidate_loss(config):
        predicted_V = candidate_solve(config).ys[:, 0, :]
        return jnp.mean((predicted_V - target_V) ** 2)

    candidate_optimizer = OptaxOptimizer(
        candidate_loss,
        optax.adam(learning_rate=3e-3),
    )

    start = time.perf_counter()
    candidate_fitted, _ = candidate_optimizer.run(
        candidate_config,
        max_steps=SWEEP_STEPS,
        chunk_size=50,
    )
    block_until_ready(candidate_fitted)
    fit_seconds = time.perf_counter() - start

    candidate_train_mse = float(candidate_loss(candidate_fitted))
    candidate_validation_config = candidate_fitted.copy()
    candidate_validation_config.initial_state.dynamics = validation_initial_state
    candidate_validation_config.noise.key = validation_noise_key
    candidate_validation = candidate_solve(candidate_validation_config)
    candidate_validation_mse = float(
        jnp.mean(
            (
                candidate_validation.ys[:, 0, :]
                - validation_target.ys[:, 0, :]
            )
            ** 2
        )
    )

    capacity_results.append(
        {
            "width": width,
            "parameters": parameter_count(candidate_correction.module),
            "train_mse": candidate_train_mse,
            "validation_mse": candidate_validation_mse,
            "fit_seconds": fit_seconds,
        }
    )

assert all(
    np.isfinite(result[key])
    for result in capacity_results
    for key in ("train_mse", "validation_mse", "fit_seconds")
)

best_validation_mse = min(
    result["validation_mse"] for result in capacity_results
)
knee_threshold = (1.0 + KNEE_TOLERANCE) * best_validation_mse
selected_capacity = next(
    result
    for result in capacity_results
    if result["validation_mse"] <= knee_threshold
)

In [ ]:

parameter_counts = np.array(
    [result["parameters"] for result in capacity_results]
)
train_errors = np.array(
    [result["train_mse"] for result in capacity_results]
)
validation_errors = np.array(
    [result["validation_mse"] for result in capacity_results]
)
fit_times = np.array(
    [result["fit_seconds"] for result in capacity_results]
)

fig, (ax_error, ax_time) = plt.subplots(1, 2, figsize=(8.1, 3.4))

ax_error.loglog(
    parameter_counts,
    train_errors,
    "o-",
    color="0.45",
    label="training",
)
ax_error.loglog(
    parameter_counts,
    validation_errors,
    "o-",
    color="#1f5a99",
    label="validation",
)
ax_error.axhline(
    knee_threshold,
    color="#1f5a99",
    linestyle=":",
    alpha=0.7,
    label="20% tolerance",
)
ax_error.scatter(
    selected_capacity["parameters"],
    selected_capacity["validation_mse"],
    s=100,
    facecolors="none",
    edgecolors="#c44e52",
    linewidths=2,
    zorder=4,
    label=f"selected: width {selected_capacity['width']}",
)
for result in capacity_results:
    ax_error.annotate(
        f"w={result['width']}",
        (result["parameters"], result["validation_mse"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=7,
    )
ax_error.set_xlabel("Trainable parameters")
ax_error.set_ylabel("Voltage trajectory MSE")
ax_error.set_title("Accuracy–capacity trade-off")
ax_error.grid(alpha=0.2, which="both")
ax_error.legend(frameon=False, fontsize=7)

ax_time.plot(parameter_counts, fit_times, "o-", color="#c44e52")
for result in capacity_results:
    ax_time.annotate(
        f"w={result['width']}",
        (result["parameters"], result["fit_seconds"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=7,
    )
ax_time.set_xscale("log")
ax_time.set_xlabel("Trainable parameters")
ax_time.set_ylabel("First-fit wall time [s]")
ax_time.set_title(f"500 steps on {jax.devices()[0].device_kind}")
ax_time.grid(alpha=0.2, which="both")

plt.tight_layout()

In this run the rule selects width 16, with 337 trainable parameters. Increasing
the width to 32 raises the count to 1,185 and takes longer, but does not improve
validation error. That is the kind of diminishing-return knee the sweep is
designed to expose.

The timing includes tracing and compilation, so it describes the complete first
fit a notebook user experiences rather than steady-state kernel throughput. It
is hardware- and software-dependent and should not be treated as a portable
benchmark. Compilation caches and system load can also make small timing
differences non-monotonic.

This single sweep gives an objective rule for this example, but not uncertainty
on the architecture choice. For a scientific analysis, repeat the comparison
across initialization and noise seeds, then evaluate the selected architecture
once on a final test set that was not used for fitting or model selection.

::: {.callout-warning}
## Interpretation and identifiability

Trajectory agreement does not prove that the MLP discovered a unique physical
law. A flexible correction can compensate for errors in fixed parameters,
coupling, observations, or initial conditions. Here those quantities are held at
their known synthetic values specifically so that the residual has an
interpretable target. In real data, compare alternative parameterizations,
inspect identifiability, and validate on held-out conditions.
:::

::: {.callout-note}
## From this controlled example to noisy observations

Common random numbers isolate model error in this synthetic experiment because
the latent forcing is known. With experimental observations, that forcing is
usually unknown. A practical loss may then average over several simulated noise
realizations or compare robust summaries rather than matching one trajectory
point by point.
:::

## Summary

This tutorial demonstrated the complete UDE workflow in TVB-Optim:

1. A mechanistic network model supplied stable, interpretable dynamics and
   explicit inter-region coupling.
2. One `EquinoxParameter` marked a shared MLP as trainable inside
   `config.dynamics`.
3. `eqx.filter_jit` compiled a full configuration containing callable Equinox
   leaves.
4. Gradients propagated through the network solver to every MLP array leaf.
5. `OptaxOptimizer` fitted the correction without a special neural-network
   optimization path.
6. Multiple nodes jointly trained the same local correction under fixed random
   forcing, which was then evaluated on validation initial conditions and a new
   noise realization.
7. A capacity sweep compared validation error, parameter count, and indicative
   first-fit runtime instead of assuming that a larger MLP is better.

The essential modeling choice is separation of responsibilities: known coupling
and slow recovery dynamics remain mechanistic, while the MLP learns only the
local voltage residual placed explicitly in the equations.

## References

- Rackauckas, C., Ma, Y., Martensen, J., et al. (2020). [Universal Differential
  Equations for Scientific Machine Learning](https://doi.org/10.48550/arXiv.2001.04385).
  *arXiv:2001.04385*.
- El-Gazzar, A., and van Gerven, M. (2025). [Universal differential equations as
  a unifying modeling language for
  neuroscience](https://doi.org/10.3389/fncom.2025.1677930). *Frontiers in
  Computational Neuroscience*, 19, 1677930.
- Chen, R. T. Q., Rubanova, Y., Bettencourt, J., and Duvenaud, D. K. (2018).
  [Neural Ordinary Differential
  Equations](https://proceedings.neurips.cc/paper/2018/hash/69386f6bb1dfed68692a24c8686939b9-Abstract.html).
  *Advances in Neural Information Processing Systems*, 31.
- FitzHugh, R. (1961). [Impulses and physiological states in theoretical models
  of nerve membrane](https://doi.org/10.1016/S0006-3495(61)86902-6).
  *Biophysical Journal*, 1(6), 445--466.
- Nagumo, J., Arimoto, S., and Yoshizawa, S. (1962). [An active pulse
  transmission line simulating nerve
  axon](https://doi.org/10.1109/JRPROC.1962.288235). *Proceedings of the IRE*,
  50(10), 2061--2070.